In [1]:
import kagglehub
# Download latest version
path = kagglehub.competition_download('ace-aiml-recruitmen-ts-26-1')
print("Path to competition files:", path)
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import mean_squared_log_error

Path to competition files: /kaggle/input/competitions/ace-aiml-recruitmen-ts-26-1


In [2]:
# Checking the available input files

import os
print(os.listdir('/kaggle/input'))

['competitions', 'datasets']


In [3]:
# Finding the attached dataset location

import os
for root, dirs, files in os.walk('/kaggle/input'):
    if files:
        print(root)
        print(files)

/kaggle/input/competitions/ace-aiml-recruitmen-ts-26-1
['sample_submission.csv', 'train.csv', 'test.csv']
/kaggle/input/datasets/shreenidhi1503/part-1
['sample_submission.csv', 'train.csv', 'test.csv']


In [4]:
# Loading the training and test datasets

train = pd.read_csv('/kaggle/input/datasets/shreenidhi1503/part-1/train.csv')
test = pd.read_csv('/kaggle/input/datasets/shreenidhi1503/part-1/test.csv')

hpd = train

In [5]:
# Viewing the first few rows
hpd.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,382,527359050,20,RL,80.0,10400,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2009,WD,Family,152000
1,835,906475070,60,RL,NaN,28698,Pave,NaN,IR2,Low,...,0,NaN,NaN,NaN,0,6,2009,WD,Abnorml,185000
2,1899,534429030,90,RL,70.0,9842,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,3,2007,WD,Normal,101800
3,679,535451170,90,RL,60.0,7200,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,6,2009,WD,Normal,90000
4,701,902109130,190,RM,63.0,7627,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,10,2009,WD,Normal,94550


In [6]:
# Checking the number of rows and columns
hpd.shape

(2344, 82)

In [7]:
# Checking the column names
hpd.columns

Index(['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area',
       'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities',
       'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Overall Qual',
       'Overall Cond', 'Year Built', 'Year Remod/Add', 'Roof Style',
       'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type',
       'Mas Vnr Area', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual',
       'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1',
       'BsmtFin Type 2', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
       'Heating', 'Heating QC', 'Central Air', 'Electrical', '1st Flr SF',
       '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath',
       'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr',
       'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Functional',
       'Fireplaces', 'Fireplace Qu', 'Garage Type', 'Garage Yr Blt',
      

In [8]:
# Checking for missing values
hpd.isnull().sum()

Order               0
PID                 0
MS SubClass         0
MS Zoning           0
Lot Frontage      393
                 ... 
Mo Sold             0
Yr Sold             0
Sale Type           0
Sale Condition      0
SalePrice           0
Length: 82, dtype: int64

In [9]:
# Checking basic statistical information
hpd.describe()

,Order,PID,MS SubClass,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,...,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,SalePrice
count,2344.000000,2.344000e+03,2344.000000,1951.000000,2344.000000,2344.00000,2344.000000,2344.000000,2344.000000,2325.000000,...,2344.000000,2344.000000,2344.000000,2344.000000,2344.000000,2344.000000,2344.000000,2344.000000,2344.000000,2344.000000
mean,1464.298635,7.143632e+08,57.308020,69.207586,10127.857509,6.06442,5.581911,1970.506826,1983.923635,98.539785,...,94.113055,47.341724,24.731655,2.422782,16.010239,2.582765,58.055034,6.184727,2007.794795,178582.207765
std,842.982263,1.886907e+08,42.802555,23.418018,8050.908132,1.38852,1.105659,30.341434,20.786287,172.239113,...,124.851870,68.037465,67.030943,24.524362,55.820882,38.324145,623.375121,2.708408,1.315117,77125.072713
min,1.000000,5.263011e+08,20.000000,21.000000,1300.000000,1.00000,1.000000,1872.000000,1950.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,12789.000000
25%,737.750000,5.284821e+08,20.000000,58.000000,7466.500000,5.00000,5.000000,1953.000000,1965.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,2007.000000,129000.000000
50%,1466.000000,5.354541e+08,50.000000,68.000000,9356.500000,6.00000,5.000000,1972.000000,1992.000000,0.000000,...,0.000000,26.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,160000.000000
75%,2195.250000,9.071921e+08,70.000000,80.000000,11484.250000,7.00000,6.000000,2000.000000,2003.000000,160.000000,...,168.000000,70.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,210000.000000
max,2930.000000,9.241520e+08,190.000000,313.000000,215245.000000,10.00000,9.000000,2010.000000,2010.000000,1290.000000,...,870.000000,742.000000,1012.000000,508.000000,576.000000,800.000000,17000.000000,12.000000,2010.000000,755000.000000


In [10]:
# Separating the features and target variable
X = hpd.drop(['SalePrice'], axis=1)
Y = hpd['SalePrice']
X = pd.get_dummies(X)
X = X.astype(float)

In [11]:
# Viewing the features and target
print(X)
print(Y)

       Order          PID  MS SubClass  Lot Frontage  Lot Area  Overall Qual  \
0      382.0  527359050.0         20.0          80.0   10400.0           7.0   
1      835.0  906475070.0         60.0           NaN   28698.0           5.0   
2     1899.0  534429030.0         90.0          70.0    9842.0           4.0   
3      679.0  535451170.0         90.0          60.0    7200.0           4.0   
4      701.0  902109130.0        190.0          63.0    7627.0           4.0   
...      ...          ...          ...           ...       ...           ...   
2339  1639.0  527226040.0        190.0         195.0   41600.0           5.0   
2340  1096.0  528290090.0         60.0           NaN    7750.0           7.0   
2341  1131.0  528490080.0         60.0          64.0    8791.0           6.0   
2342  1295.0  902109160.0         50.0          81.0    7308.0           5.0   
2343   861.0  907227110.0         85.0          60.0    7200.0           5.0   

      Overall Cond  Year Built  Year Re

In [12]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

In [13]:
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_log_error

model = XGBRegressor(
    n_estimators=2500,
    learning_rate=0.02,
    max_depth=5,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=1.5,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    np.log1p(Y_train),
    eval_set=[(X_test, np.log1p(Y_test))],
    verbose=False
)

training_data_prediction = np.expm1(
    model.predict(X_train)
)

test_data_prediction = np.expm1(
    model.predict(X_test)
)

training_data_prediction = np.maximum(training_data_prediction, 0)
test_data_prediction = np.maximum(test_data_prediction, 0)

msle = mean_squared_log_error(
    Y_test,
    test_data_prediction
)

print("MSLE:", msle)

MSLE: 0.013277576304972172


In [14]:
print(X.shape,X_train.shape,X_test.shape)

(2344, 302) (1875, 302) (469, 302)


In [15]:
training_data_prediction=np.expm1(model.predict(X_train))
print(training_data_prediction)

[117809.22  129279.3   124427.35  ... 127578.195 164843.4   357131.1  ]


In [16]:
# Making predictions on validation data
# Predict log prices and convert back
test_data_prediction = np.expm1(model.predict(X_test))
print(test_data_prediction)

[298858.8   122007.31  320900.12  141646.7   153630.38  165547.
  98840.53  134354.52  172904.11  176546.05  109846.96  175431.95
 116315.266 111793.25  213602.61  136215.6   143353.58  100831.734
 132889.95  211701.97   92747.06  122235.58  154802.69  169715.3
 175150.1   242699.12  173465.17  235220.6   137242.58  176022.03
 180778.42  182542.92  118393.21  220465.94  284376.75  196725.83
 150745.67  131976.81  158662.62  157985.73  154073.92  143068.4
 133708.    164512.19   85602.38  409043.94  169099.42  254807.11
 163141.52  182407.19  107096.74  157996.28  125968.66  154360.86
 184574.08  187862.28  126103.04  339059.38  101717.59  134989.5
 108758.19  164120.89   82816.625 114451.36  455762.3   119922.66
 128787.195 146564.19  151098.6   127911.27  153188.83  125143.29
 155747.45  127800.31  160808.56   54480.465 208870.31  221595.36
 277396.28  264761.5   241637.45   89028.39  160366.27  186099.25
 109544.31  108051.15  128761.04  146278.06  109243.44  125255.766
 179511.4   3

In [17]:
# Prepare the actual competition test data

X_competition = test.drop(columns=['SalePrice'], errors='ignore')
X_competition = pd.get_dummies(X_competition)

# Make the columns exactly the same as the training data
X_competition = X_competition.reindex(columns=X.columns, fill_value=0)

# Predict the actual competition test data
competition_prediction = np.expm1(model.predict(X_competition))

# Create submission
submission = pd.DataFrame({
    'PID': test['PID'],
    'SalePrice': competition_prediction
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission)
print("Submission shape:", submission.shape)

           PID      SalePrice
0    903427090  168595.250000
1    527450460  108662.281250
2    908128100  185933.750000
3    907135180  121080.351562
4    910200080  113612.250000
..         ...            ...
581  916455110  181483.031250
582  528176030  456140.187500
583  528188020  178694.437500
584  908102160  128463.843750
585  908102260  151065.750000

[586 rows x 2 columns]
Submission shape: (586, 2)


In [18]:
from sklearn.metrics import mean_squared_log_error
msle = mean_squared_log_error(
    Y_test,
    np.maximum(test_data_prediction, 0)
)
print("MSLE:", msle)

MSLE: 0.013277576304972172
